# Gabarito — Inequações Trigonométricas

Este notebook traz a resolução comentada dos exercícios de [`0508a_inequacoes_trigonometricas.ipynb`](0508a_inequacoes_trigonometricas.ipynb).

⚠️ **Antes de olhar aqui:** tente resolver os exercícios sozinho no notebook original e use as células de verificação (`assert`) para conferir suas respostas. Volte para cá só se travar ou quiser comparar sua solução.

In [1]:
import math
from collections.abc import Callable

import numpy as np


def intervalos_da_mascara(xs: np.ndarray, mascara: np.ndarray) -> list[tuple[float, float]]:
    """Converte uma máscara booleana nos subintervalos [início, fim] onde ela é True (função reaproveitada de 0508a_inequacoes_trigonometricas)."""
    intervalos = []
    inicio = None
    for i, dentro in enumerate(mascara):
        if dentro and inicio is None:
            inicio = xs[i]
        elif not dentro and inicio is not None:
            intervalos.append((inicio, xs[i - 1]))
            inicio = None
    if inicio is not None:
        intervalos.append((inicio, xs[-1]))
    return intervalos


def encontrar_intervalos_solucao(
    funcao: Callable[[np.ndarray], np.ndarray],
    comparador: Callable[[np.ndarray, float], np.ndarray],
    valor_m: float,
    intervalo: tuple[float, float],
    n_pontos: int = 4000,
) -> list[tuple[float, float]]:
    """Varre o intervalo numa grade fina e devolve os subintervalos onde comparador(funcao(x), valor_m) é verdadeiro (função reaproveitada de 0508a_inequacoes_trigonometricas)."""
    xs = np.linspace(intervalo[0], intervalo[1], n_pontos)
    ys = funcao(xs)
    mascara = comparador(ys, valor_m) & ~np.isnan(ys)
    return intervalos_da_mascara(xs, mascara)


def tangente_com_assintotas(x: np.ndarray) -> np.ndarray:
    """Calcula tg(x), com NaN perto das assíntotas (função reaproveitada de 0508a_inequacoes_trigonometricas)."""
    cos_x = np.cos(x)
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.where(np.abs(cos_x) < 0.03, np.nan, np.sin(x) / cos_x)


def cossecante(x: np.ndarray) -> np.ndarray:
    """Calcula cossec(x) = 1/sen(x), com NaN perto das assíntotas (função reaproveitada de 0508a_inequacoes_trigonometricas)."""
    sen_x = np.sin(x)
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.where(np.abs(sen_x) < 0.03, np.nan, 1 / sen_x)

## Básico

**Exercício 1.** Resolva a inequação sen x > 0 no intervalo [0, 2π).

In [2]:
# sen x > 0 corresponde ao arco "de cima" do corte horizontal y=0 no ciclo --
# o primeiro e o segundo quadrantes
intervalos_ex1 = encontrar_intervalos_solucao(np.sin, np.greater, 0, (0, 2 * math.pi))
print(intervalos_ex1)

[(np.float64(0.0015711891240759155), np.float64(3.1408070590277553))]


**Exercício 2.** Resolva a inequação cos x < 1/2 no intervalo [0, 2π).

In [3]:
# cos x < 1/2 corresponde à região à esquerda do corte vertical x=1/2 no ciclo
intervalos_ex2 = encontrar_intervalos_solucao(np.cos, np.less, 0.5, (0, 2 * math.pi))
print(intervalos_ex2)

[(np.float64(1.0479831457586357), np.float64(5.2352021614209505))]


## Intermediário

**Exercício 3.** Resolva cossec x > 2 no intervalo [0, 2π), reduzindo à inequação equivalente em seno.

In [4]:
# cossec x > 2 (positiva) só é possível com sen x > 0, e nesse caso
# cossec x > 2 <=> sen x < 1/2 -- ou seja, 0 < sen x < 1/2
intervalos_ex3 = encontrar_intervalos_solucao(cossecante, np.greater, 2, (0, 2 * math.pi))
print(intervalos_ex3)

[(np.float64(0.03142378248151831), np.float64(0.5232059783172799)), (np.float64(2.619172269834551), np.float64(3.1109544656703125))]


**Exercício 4.** Resolva o sistema tg x > 0 e sen x < 0,5 no intervalo [0, 2π), encontrando a interseção.

In [5]:
# interseção: máscara booleana com as duas condições ao mesmo tempo
xs_ex4 = np.linspace(0, 2 * math.pi, 4000)
mascara_ex4 = (tangente_com_assintotas(xs_ex4) > 0) & (np.sin(xs_ex4) < 0.5)
intervalos_ex4 = intervalos_da_mascara(xs_ex4, mascara_ex4)
print(intervalos_ex4)

[(np.float64(0.0015711891240759155), np.float64(0.5232059783172799)), (np.float64(3.142378248151831), np.float64(4.682143589746229))]


## Desafio

**Exercício 5.** A potência de um painel solar ao longo do dia é P(h) = 500·sen(π/12·(h−6)) watts, para 6 ≤ h ≤ 18. Encontre os intervalos de horário em que P(h) > 400 W.

In [6]:
def potencia_painel(h: np.ndarray) -> np.ndarray:
    """Potência (em watts) recebida pelo painel solar às h horas."""
    return 500 * np.sin(math.pi / 12 * (h - 6))


intervalos_ex5 = [
    (round(inicio, 2), round(fim, 2))
    for inicio, fim in encontrar_intervalos_solucao(potencia_painel, np.greater, 400, (6, 18))
]
print(intervalos_ex5)

[(np.float64(9.54), np.float64(14.46))]


**Exercício 6.** Resolva a inequação com múltiplo arco sen(2x) > 1/2 no intervalo [0, 2π).

In [7]:
# tratamos 2x como a incógnita, resolvendo em (0, 4*pi), e só então dividimos
# CADA extremidade por 2
intervalos_2x_ex6 = encontrar_intervalos_solucao(np.sin, np.greater, 0.5, (0, 4 * math.pi))
intervalos_ex6 = [(round(inicio / 2, 4), round(fim / 2, 4)) for inicio, fim in intervalos_2x_ex6]
print(intervalos_ex6)

[(np.float64(0.2624), np.float64(1.3088)), (np.float64(3.4048), np.float64(4.4496))]
